In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras.models import Model, load_model
import os
import glob
from keras.datasets import fashion_mnist
from tensorflow.keras.initializers import GlorotUniform, HeNormal, RandomNormal
from numpy.random import randn, randint
import soundfile as sf
from IPython.display import Audio

In [ ]:
# Load ESC-50 dataset
def load_esc50_data(dataset_path, sample_rate=22050):
    audio_files = glob.glob(os.path.join(dataset_path, "*.wav"))
    data, raw_audio = [], []
    for file in audio_files:
        y, sr = librosa.load(file, sr=sample_rate)
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr)
        mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
        data.append(mel_spec)
        raw_audio.append((y, sr))
    return np.array(data), raw_audio

import kagglehub

# Download latest version
path = kagglehub.dataset_download("mmoreaux/environmental-sound-classification-50")

print("Path to dataset files:", path)


In [ ]:
# Define dataset path (update with correct path)
dataset_path = path+"/audio/audio"
data, raw_audio = load_esc50_data(dataset_path)

In [ ]:
# A function to normalize data
def normalize_data(data, s, m):
    data = np.float32(data)

    if(m==0):
        m = np.min(data)
    if(s==0):
        s = np.max(data) - np.min(data)

    norm_data = (data - m) / s
    norm_data = (norm_data -0.5) *2.
    norm_data = np.clip(norm_data, -1, 1) # just a sanity check
    return norm_data, s, m

# A function to denormalize data
def denormalize_data(norm_data, scale, mean):

    data = norm_data/2 + 0.5
    data = (data * scale) + mean
    return data

In [ ]:
# Normalize the data
norm_data, data_scale, data_mean = normalize_data(data, 0, 0)
print('Data shape:', data.shape)

print('Normalized Data shape:', norm_data.shape)
print(norm_data.dtype)

In [ ]:
# Function to convert a spectrogram to audio

def spectrogram_to_audio(mel_spec_db, sr=22050, n_fft=2048, hop_length=512, n_mels=128, griffin_lim_iters=64):

    mel_spec = librosa.db_to_power(mel_spec_db)
    stft_spec = librosa.feature.inverse.mel_to_stft(mel_spec, sr=sr, n_fft=n_fft)
    audio = librosa.griffinlim(stft_spec, n_iter=griffin_lim_iters, hop_length=hop_length, win_length=n_fft)
    audio = audio / np.max(np.abs(audio))
    audio = audio.astype(np.float32)

    return audio


sample_rate=22050

In [ ]:
# A function to sample a batch of real data
def sample_real_data(data, numdata):
    random_pos = np.random.choice(len(data), numdata, replace=False)
    X = data[random_pos]
    y = np.ones((numdata, 1)) # the default label for real data is one
    return X, y, random_pos


def get_real_audio(audio, pos):
    selected_audio = []
    for p in pos:
        selected_audio.append(audio[p])
    return selected_audio

In [ ]:
# a function to visualize a batch of data
def visualizeSamples(data, pos, numdata):
    if(len(np.shape(data)) == 4):
        data = np.squeeze(data)

    fig, axes = plt.subplots(1, numdata, figsize=(15, numdata))

    for i, ax in enumerate(axes):
        librosa.display.specshow(data[i, :, :], sr=22050, x_axis='time', y_axis='mel', cmap='viridis', ax=ax)
        ax.set_title(f"Spectrogram of sample: {pos[i]}")
    plt.tight_layout()
    plt.show()

In [ ]:
# A function to enable hearing the sound
def showAudio(audio, pos, numdata):
    # Play selected input samples
    for i, (y, sr) in enumerate(audio):
        print(f"Playing sample {pos[i]}")
        display(Audio(y, rate=sr))
        if(i == numdata-1):
            return

In [ ]:
# Select some random data to be used as a reference in the visualization
numdata = 5
samp_data, samp_label, samp_pos = sample_real_data(data, numdata)
# get the corresponding audio
samp_audio = get_real_audio(raw_audio, samp_pos)

# Visualize selected input images
visualizeSamples(samp_data, samp_pos, numdata)

showAudio(samp_audio, samp_pos, numdata)

In [ ]:
# Build Variational Autoencoder (VAE)
.... Add your code here ...

In [ ]:
# Compile and train VAE
... Add your code here ...

In [ ]:
# Plot training and validation loss
... Add your code here ...

In [ ]:
# Generate reconstructed images <-- This is to check that the latent space does make sense
... Add your code here ...

In [ ]:
# Try and generate new data with the decoder
# To this purpose we need to start from random noise in the latent space

... Add your code here ...

In [ ]:
# Build, compile and train a GAN
... Add your code here ...

In [ ]:
# Try and generate some samples with the GAN
... Add your code here ...